In [60]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
import numpy as np
import scipy
from scipy.spatial.distance import cdist

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

**<font color="green"> Part 1: reading numpy files</font>**

In [61]:
edgeindex = np.load('../embeddings/cora_edge_index.npy')
embeddings = np.load('../embeddings/cora_embeddings.npy')

# printing them to see size...
np.set_printoptions(threshold=1000)
print(f'EdgeIndex:\n{edgeindex}')
#np.set_printoptions(threshold=np.inf)
print(f'Embeddings:\n{embeddings}')
np.set_printoptions(threshold=1000)

eiM, eiN = edgeindex.shape
embM, embN = embeddings.shape

print(f'edgeindex shape: {eiM} x {eiN}')
print(f'embeddings shape: {embM} x {embN}')

EdgeIndex:
[[   0    0    0 ... 2707 2707 2707]
 [ 633 1862 2582 ...  598 1473 2706]]
Embeddings:
[[-12.570496  -11.871372  -13.102854  ... -13.445804  -18.870369
  -13.427263 ]
 [-17.313646  -27.344126  -20.099285  ...   9.469983  -43.64191
  -26.408186 ]
 [-21.26671   -26.189857  -15.5048065 ...   4.443268  -41.934475
  -26.266088 ]
 ...
 [ -8.456446   -8.01565   -15.471559  ... -14.292109   -2.1965485
  -14.189591 ]
 [-17.272263  -14.681144   -6.2844157 ...  -6.099689  -17.407452
  -21.187057 ]
 [-15.650577  -12.82992    -7.184733  ...  -7.125296  -16.254532
  -18.21273  ]]
edgeindex shape: 2 x 10556
embeddings shape: 2708 x 7


**<font color="green"> Part 2: creating the adjacency matrix</font>**

In [62]:
adjMat = np.zeros(embM*embM).reshape(embM, embM)

for i in range(eiN):
    x = edgeindex[0][i]
    y = edgeindex[1][i]
    adjMat[x][y] = 1

print(adjMat)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 1. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 1. 0.]]


**<font color="green">Part 3: Calculating Pairwise Distances between all nodes</font>**

*<font color = "blue">I think I use the embedding 1-7 numbers?</font>*

In [63]:
distMat = np.zeros(embM*embM).reshape(embM, embM)

distMat = cdist(embeddings, embeddings, metric='euclidean')

print(distMat)

[[ 0.         55.71133859 45.5548731  ... 22.75613732 15.37738282
  11.92557015]
 [55.71133859  0.         13.49139888 ... 59.1490818  48.26070859
  50.83601269]
 [45.5548731  13.49139888  0.         ... 52.49209396 37.68916998
  40.58633498]
 ...
 [22.75613732 59.1490818  52.49209396 ...  0.         24.72615298
  22.12137172]
 [15.37738282 48.26070859 37.68916998 ... 24.72615298  0.
   4.47874623]
 [11.92557015 50.83601269 40.58633498 ... 22.12137172  4.47874623
   0.        ]]


**<font color="green">Part 4: Identifying knns for each node (gd, indices)(argsort)</font>**

In [64]:
# # This is the 10 closest nodes to node0
# sorted_indices = np.argsort(distMat[0])
# closest_nodes = sorted_indices[1:11]
# print("Indices of the 10 closest nodes:", closest_nodes)



**<font color="green">Part 5: Adding Edges</font>**

In [65]:
numberOfKNNs = 10
count = 0
#edgesToAdd = np.zeros(embM*numberOfKNNs).reshape(embM, numberOfKNNs)

# adding edges directly in loop
for i in range(embM):
    sorted_indices = np.argsort(distMat[i])
    indexOfKNNs = sorted_indices[1:numberOfKNNs+1]
    for index in indexOfKNNs:
        if adjMat[i][index] == 0:
            count += 1
            adjMat[i][index] = 1
            adjMat[index][i] = 1

print(f'Number of Edges Added: {count}')
# 17463 edges added for 10 KNNs

Number of Edges Added: 17463


**<font color="green">Part 6: See change in accuracy???</font>**